In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# =========================================================
# Configuration
# =========================================================
BASE_DIR = Path(".")
MODELS = ["ArcFace", "ElasticFace", "SwinTransformer_S"]
FMR_LEVELS = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

# =========================================================
# Evaluation function
# =========================================================
def evaluate_model(model_name, base_dir=BASE_DIR):
    model_dir = base_dir / model_name / "all_sections"

    sar_path = model_dir / "combined_sar.npy"
    impostor_path = model_dir / "combined_impostor.npy"

    if not sar_path.exists():
        raise FileNotFoundError(f"Missing file: {sar_path}")
    if not impostor_path.exists():
        raise FileNotFoundError(f"Missing file: {impostor_path}")

    sar_scores = np.load(sar_path)
    impostor_scores = np.load(impostor_path)

    labels = np.concatenate([
        np.ones_like(sar_scores),
        np.zeros_like(impostor_scores)
    ])
    scores = np.concatenate([sar_scores, impostor_scores])

    print(f"[INFO] Computing ROC for {model_name}...")
    fmr, sar, thresholds = roc_curve(labels, scores)

    results = {}
    selected_fmrs = []
    selected_sars = []

    for fmr_target in FMR_LEVELS:
        idx = np.argmin(np.abs(fmr - fmr_target))
        results[str(fmr_target)] = {
            "threshold": float(thresholds[idx]),
            "SAR": float(sar[idx]),
            "FMR(actual)": float(fmr[idx])
        }
        selected_fmrs.append(float(fmr[idx]))
        selected_sars.append(float(sar[idx]))

    with open(model_dir / "multiFMR_results.json", "w") as f:
        json.dump(results, f, indent=4)

    print(f"[INFO] Saved: {model_dir / 'multiFMR_results.json'}")
    return results, fmr, sar, selected_fmrs, selected_sars

# =========================================================
# Main plotting
# =========================================================
all_results = {}

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

for model in MODELS:
    results, fmr, sar, selected_fmrs, selected_sars = evaluate_model(model)
    all_results[model] = results

    ax.plot(fmr, sar, linewidth=2, label=model)
    ax.scatter(selected_fmrs, selected_sars, s=28) # can be remove to see a smooth graph

ax.set_xscale("log")
ax.set_xlabel("FMR", fontsize=12)
ax.set_ylabel("SAR", fontsize=12)
ax.set_title("SAR–FMR Comparison of Models", fontsize=15, pad=10)

ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend(fontsize=11, frameon=True)

plt.tight_layout()

output_png = BASE_DIR / "SAR_FMR_comparison.png"
plt.savefig(output_png, dpi=300, bbox_inches="tight")
plt.show()

output_json = BASE_DIR / "sar_fmr_comparison.json"
with open(output_json, "w") as f:
    json.dump(all_results, f, indent=4)

print(f"[DONE] Saved figure: {output_png}")
print(f"[DONE] Saved summary JSON: {output_json}")

[INFO] Computing ROC for ArcFace...
